# EX06 — KNN klassifikaator rinnavähi diagnostika andmestikul

Eesmärk: rakendada k-lähimate naabrite (KNN) klassifikaatorit ning hinnata mudeli võimet tuvastada pahaloomulisi kasvajaid.

In [1]:
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score

## 1. Andmestiku laadimine

In [2]:
data = load_breast_cancer()

X = data.data
y = data.target
feature_names = data.feature_names
target_names = data.target_names

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Classes: {target_names}")
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

X shape: (569, 30)
y shape: (569,)
Classes: ['malignant' 'benign']
Class distribution: {np.int64(0): np.int64(212), np.int64(1): np.int64(357)}


## 2. Andmete kirjeldamine (DataFrame)

In [3]:
df = pd.DataFrame(X, columns=feature_names)
df['diagnosis'] = target_names[y]

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         5

In [4]:
df.describe()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,0.062798,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,0.007060,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,0.057700,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,0.061540,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,0.066120,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,0.097440,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


In [5]:
print("Missing values:")
print(df.isnull().sum().sum())

print("\nClass distribution:")
print(df['diagnosis'].value_counts())

Missing values:
0

Class distribution:
diagnosis
benign       357
malignant    212
Name: count, dtype: int64


## 3. Treening- ja testandmete jagamine

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"y_test distribution:  {dict(zip(*np.unique(y_test, return_counts=True)))}")

X_train: (455, 30), X_test: (114, 30)
y_train distribution: {np.int64(0): np.int64(170), np.int64(1): np.int64(285)}
y_test distribution:  {np.int64(0): np.int64(42), np.int64(1): np.int64(72)}


## 4. Andmete skaleerimine (StandardScaler)

KNN kasutab eukleidilist kaugust, mistõttu on oluline, et kõik tunnused oleksid samal skaalal. StandardScaler teisendab iga tunnuse nii, et keskmine = 0 ja standardhälve = 1.

In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Before scaling — X_train[:3, :3]:")
print(X_train[:3, :3])
print("\nAfter scaling — X_train_scaled[:3, :3]:")
print(X_train_scaled[:3, :3])

Before scaling — X_train[:3, :3]:
[[ 10.32  16.35  65.31]
 [ 20.18  19.54 133.8 ]
 [ 10.66  15.15  67.49]]

After scaling — X_train_scaled[:3, :3]:
[[-1.07200079 -0.6584246  -1.0880801 ]
 [ 1.74874285  0.06650173  1.75115682]
 [-0.97473376 -0.93112416 -0.99770871]]


## 5. KNN implementeerimine käsitsi

In [8]:
def my_knn_predict_single(X_train, y_train, x_new, k=5):
    """
    Ennusta ühe uue punkti klass KNN abil.
    Sammud:
    1) Arvuta eukleidiline kaugus x_new ja kõigi X_train punktide vahel
    2) Leia k lähimat treeningpunkti
    3) Võta nende punktide klassid
    4) Tee enamus-hääletus (Counter)
    """
    distances = np.linalg.norm(X_train - x_new, axis=1)
    nearest_indices = np.argsort(distances)[:k]
    nearest_labels = y_train[nearest_indices]
    most_common = Counter(nearest_labels.tolist()).most_common(1)[0][0]
    return most_common


def my_knn_predict(X_train, y_train, X_new, k=5):
    """
    Ennusta mitme punkti klassid, kutsudes my_knn_predict_single iga punkti jaoks
    """
    return np.array([
        my_knn_predict_single(X_train, y_train, x, k=k)
        for x in X_new
    ])

## 6. Ennustamine ja hindamine

In [9]:
my_preds = my_knn_predict(X_train_scaled, y_train, X_test_scaled, k=5)
score = recall_score(y_test, my_preds, pos_label=0)

print(f"Malignant recall (k=5): {score:.4f}")

Malignant recall (k=5): 0.9286


## Analüüs: k väärtuse mõju

In [10]:
k_values = range(1, 22, 2)
recalls = []

for k in k_values:
    preds_k = my_knn_predict(X_train_scaled, y_train, X_test_scaled, k=k)
    r = recall_score(y_test, preds_k, pos_label=0)
    recalls.append(r)
    print(f"k={k:2d}  recall(malignant)={r:.4f}")

best_idx = np.argmax(recalls)
print(f"\nBest k={list(k_values)[best_idx]} with recall={recalls[best_idx]:.4f}")

k= 1  recall(malignant)=0.9286
k= 3  recall(malignant)=0.9524
k= 5  recall(malignant)=0.9286
k= 7  recall(malignant)=0.9286
k= 9  recall(malignant)=0.9286
k=11  recall(malignant)=0.9286
k=13  recall(malignant)=0.9286
k=15  recall(malignant)=0.9286
k=17  recall(malignant)=0.9524
k=19  recall(malignant)=0.9286
k=21  recall(malignant)=0.9048

Best k=3 with recall=0.9524


## 7. Kokkuvõte

In [11]:
summary = (
    "StandardScaler normalizes each feature to zero mean and unit variance. "
    "This is critical for KNN because it relies on Euclidean distance, "
    "and features with larger scales would otherwise dominate the distance calculation. "
    "For example, the 'mean area' feature ranges from ~140 to ~2500, while 'mean smoothness' ranges from ~0.05 to ~0.16; "
    "without scaling, area would almost entirely determine nearest neighbors. "
    "A small k (e.g., k=1) produces highly flexible decision boundaries that capture noise in the training data, "
    "leading to high variance and potential overfitting. "
    "A large k (e.g., k=15+) smooths out the decision boundary, reducing variance but increasing bias, "
    "which can cause the model to miss subtle patterns and underfit. "
    "k=5 provides a good balance for this dataset. "
    "For detecting malignant tumors, recall is the most important metric because "
    "a false negative (missing a malignant case) is far more dangerous than a false positive. "
    "Our model achieves high recall for the malignant class, "
    "meaning it successfully identifies most malignant cases in the test set."
)

print(summary)

StandardScaler normalizes each feature to zero mean and unit variance. This is critical for KNN because it relies on Euclidean distance, and features with larger scales would otherwise dominate the distance calculation. For example, the 'mean area' feature ranges from ~140 to ~2500, while 'mean smoothness' ranges from ~0.05 to ~0.16; without scaling, area would almost entirely determine nearest neighbors. A small k (e.g., k=1) produces highly flexible decision boundaries that capture noise in the training data, leading to high variance and potential overfitting. A large k (e.g., k=15+) smooths out the decision boundary, reducing variance but increasing bias, which can cause the model to miss subtle patterns and underfit. k=5 provides a good balance for this dataset. For detecting malignant tumors, recall is the most important metric because a false negative (missing a malignant case) is far more dangerous than a false positive. Our model achieves high recall for the malignant class, me